# StairNav-HM3D Colab

Notebook chính cho project **robot giao hàng trong tòa nhà dùng HM3D + Habitat-Sim**.

Luồng chạy:

```text
HM3D scene
  -> Habitat-Sim RGB/depth/semantic observation
  -> tạo delivery episodes tiếng Việt
  -> train/test command model
  -> lưu model
  -> load model
  -> robot hỏi đáp trước khi quyết định di chuyển
  -> Habitat pathfinder kiểm tra đường đi khả thi
```

Notebook này đã bỏ phần graph MVP. Từ đây trở đi, simulator chính là **Habitat-Sim trên HM3D**.

## 1. Clone repo hoặc mount Drive

Nếu chạy từ GitHub, dùng cell dưới và sửa URL repo của bạn.

In [ ]:
# Cách 1: clone GitHub
# !git clone https://github.com/YOUR_USERNAME/stairnav-llm.git

# Cách 2: nếu đã upload folder vào /content/stairnav_llm thì không cần clone.

import os
from pathlib import Path

candidates = [
    Path.cwd(),
    Path('/content/stairnav_llm'),
    Path('/content/stairnav-llm'),
    Path('/content/stairnav-llm/stairnav_llm'),
]

PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / 'requirements.txt').exists() and (candidate / 'src').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError('Không tìm thấy project root. Hãy clone repo hoặc upload/unzip stairnav_llm trước.')

os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('Files:')
print([p.name for p in PROJECT_ROOT.iterdir()])


## 2. Cài thư viện Python cho phần ML/dialogue

Cell này cài phần nhẹ. Habitat-Sim sẽ cài riêng bằng conda ở bước sau.

In [ ]:
!pip install -q -r requirements.txt
print("Core Python dependencies installed.")

## 3. Cài Habitat-Sim trên Colab

Habitat-Sim thường ổn nhất qua conda. Cell `condacolab.install()` sẽ restart runtime; sau restart, chạy lại notebook từ bước clone/cd rồi chạy tiếp cell cài Habitat-Sim.

In [ ]:
# Chạy cell này một lần nếu Colab chưa có conda.
# Sau khi chạy, runtime sẽ restart.
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# Sau khi runtime restart, chạy cell này.
!conda install -y -c conda-forge -c aihabitat habitat-sim

import habitat_sim
print("Habitat-Sim OK")

## 4. Test Habitat-Sim bằng test scenes

Trước khi tải HM3D, phải kiểm tra Habitat-Sim chạy được đã.

In [ ]:
!python -m habitat_sim.utils.datasets_download \
  --uids habitat_test_scenes \
  --data-path data/

print("Downloaded Habitat test scenes.")

## 5. Nhập Matterport token và tải HM3D minival

HM3D cần quyền academic/non-commercial từ Matterport. Không đưa token lên GitHub.

Bắt đầu bằng `hm3d_minival_v0.2`, vì nhẹ hơn train full rất nhiều.

In [ ]:
import os
from getpass import getpass

os.environ["MATTERPORT_TOKEN_ID"] = getpass("Matterport token ID: ")
os.environ["MATTERPORT_TOKEN_SECRET"] = getpass("Matterport token secret: ")
print("Token variables set for this Colab session.")

In [ ]:
!python -m habitat_sim.utils.datasets_download \
  --username "$MATTERPORT_TOKEN_ID" \
  --password "$MATTERPORT_TOKEN_SECRET" \
  --uids hm3d_minival_v0.2 \
  --data-path data/

## 6. Tìm một HM3D scene để chạy

In [ ]:
from pathlib import Path

hm3d_root = Path("data/scene_datasets/hm3d")
scene_dirs = sorted((hm3d_root / "minival").glob("[0-9]*-*"))
print("Number of HM3D minival scenes:", len(scene_dirs))

if not scene_dirs:
    raise RuntimeError("Không tìm thấy scene HM3D. Kiểm tra lại bước download.")

scene_dir = scene_dirs[0]
scene_id = next(scene_dir.glob("*.basis.glb"))
scene_config = hm3d_root / "hm3d_annotated_basis.scene_dataset_config.json"

print("scene_id:", scene_id)
print("scene_config:", scene_config)

## 7. Import source code của project

In [ ]:
import sys
from pathlib import Path

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from hm3d_habitat_adapter import HabitatHM3DConfig, HabitatHM3DSimulator
from hm3d_dataset_builder import sample_delivery_episodes, save_jsonl, load_jsonl
from command_model import train_command_models, predict_command
from dialogue_policy import DeliveryIntent, clarification_question, apply_user_clarification

print("Imported project modules from", SRC_DIR)

## 8. Test Habitat adapter với RGB/depth/semantic observation

In [ ]:
cfg = HabitatHM3DConfig(
    scene_id=str(scene_id),
    scene_dataset_config=str(scene_config),
    width=320,
    height=240,
)

sim = HabitatHM3DSimulator(cfg)
obs = sim.reset()

print("Agent position:", obs.agent_state.position)
print("RGB shape:", None if obs.rgb is None else obs.rgb.shape)
print("Depth shape:", None if obs.depth is None else obs.depth.shape)
print("Semantic shape:", None if obs.semantic is None else obs.semantic.shape)

sim.close()

## 9. Tạo subset HM3D delivery episodes

Ở bước này ta dùng HM3D pathfinder để sample start/goal navigable. Mỗi episode được gắn câu lệnh tiếng Việt.

In [ ]:
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "outputs" / "models"
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

sim = HabitatHM3DSimulator(cfg)
episodes = sample_delivery_episodes(
    sim,
    scene_id=str(scene_id),
    count=80,
    seed=7,
    min_distance=2.0,
    clarification_ratio=0.25,
)
sim.close()

episode_file = DATA_DIR / "stairnav_hm3d_minival_80.jsonl"
save_jsonl(episodes, episode_file)

print("Saved episodes:", episode_file)
print("Count:", len(episodes))
print(episodes[0])

## 10. Train/test command model và lưu model

Model baseline này học từ câu lệnh tiếng Việt để dự đoán:

- có cần hỏi lại không
- item cần giao
- loại mô tả goal

Đây chưa phải policy navigation cuối cùng, nhưng là model train/test/save rõ ràng để bắt đầu experiment.

In [ ]:
result = train_command_models(
    episode_jsonl=episode_file,
    output_dir=MODEL_DIR,
)

print("Model saved to:", result["model_file"])
print("Metrics saved to:", result["metrics_file"])

for label, metrics in result["metrics"].items():
    print("
===", label, "===")
    print(metrics)

## 11. Lưu model vào Google Drive

Cell này giúp giữ model sau khi Colab tắt runtime.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_OUT = Path('/content/drive/MyDrive/stairnav_llm_outputs')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

!cp -r outputs/models "$DRIVE_OUT/"
!cp -r data "$DRIVE_OUT/"

print("Saved artifacts to", DRIVE_OUT)

## 12. Load model đã lưu và dự đoán câu lệnh mới

In [ ]:
model_file = MODEL_DIR / "command_model.joblib"

commands = [
    "Đem tài liệu tới khu văn phòng ở cuối hành lang.",
    "Giao gói hàng này giúp tôi.",
    "Mang laptop đến phòng gần cầu thang, nếu đường bị chặn thì chọn lối khác.",
]

for command in commands:
    print("COMMAND:", command)
    print(predict_command(model_file, command))
    print()

## 13. Robot hỏi đáp trước khi quyết định di chuyển

Nếu command thiếu thông tin, robot hỏi trước. Nếu đủ thông tin, mới chuyển sang bước chọn đường/di chuyển.

In [ ]:
intent = DeliveryIntent(item=None, destination=None, recipient=None)
question = clarification_question(intent)
print("Robot:", question)

intent = apply_user_clarification(intent, "phòng 503")
question = clarification_question(intent)
print("Robot:", question)

intent = apply_user_clarification(intent, "tài liệu seminar")
question = clarification_question(intent)
print("Intent cuối:", intent)
print("Robot đã đủ thông tin để quyết định di chuyển:", question is None)

## 14. Kiểm tra đường đi khả thi bằng Habitat pathfinder

Đây là bước test navigation ở mức Habitat: dùng geodesic distance giữa start và goal. Giai đoạn tiếp theo sẽ thay oracle pathfinder bằng policy điều khiển action `move_forward`, `turn_left`, `turn_right`.

In [ ]:
rows = load_jsonl(episode_file)
sample = rows[0]

sim = HabitatHM3DSimulator(cfg)
distance = sim.geodesic_distance(sample["start_position"], sample["goal_position"])
sim.close()

print("Instruction:", sample["instruction_vi"])
print("Start:", sample["start_position"])
print("Goal:", sample["goal_position"])
print("Geodesic distance:", distance)
print("Reachable:", distance < float("inf"))

## 15. Bước tiếp theo để thành experiment nghiên cứu

Sau notebook này, bạn sẽ mở rộng:

1. Tăng số scene HM3D: từ 1 scene lên 10-20 scene minival/val.
2. Tạo 1,000+ episode tiếng Việt.
3. Thêm ASR bằng `faster-whisper` cho voice command.
4. Thay model baseline TF-IDF bằng PhoBERT/Qwen embedding hoặc LoRA nhỏ.
5. Thêm visual detector/OCR từ RGB frame.
6. Thay pathfinder oracle bằng Habitat-Lab navigation policy.
7. So sánh: text-only vs voice, map-only vs vision+map, direct LLM vs dialogue-before-move.